# Assignment 2: The "Smart Labeling Pipeline" Challenge

**Total Marks: 20**

Build a cost-effective, high-quality labeling pipeline using human annotation, programmatic rules, and LLMs.

This notebook implements an end-to-end smart labeling pipeline to:
1. Establish gold standard through human annotation and measure inter-annotator agreement (6 marks)
2. Label data programmatically using weak supervision (Snorkel) (6 marks)
3. Optimize labeling budget using active learning (5 marks)
4. Leverage LLMs for bulk labeling and detect hallucinations (e.g. noisy labels) (3 marks)

## Setup and Imports

In [ ]:
!pip install snorkel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 3.6 MB/s eta 0:00:00


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.model import LabelModel
from statsmodels.stats.inter_rater import fleiss_kappa
import google.generativeai as genai
import time
from pathlib import Path
import re

print("All Imports Scuccessful")

All Imports Scuccessful


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Task 1: The Human as Annotator (6 Marks)

**Objective:** Establish a "Gold Standard" dataset and measure human consensus.

### Part 1.1: Parse Annotator CSV Files

After annotating the first 100 reviews, export annotations from three annotators (A, B, C) as CSV files.
Parse these CSV files into clean DataFrames for analysis.

In [ ]:
def parse_annotator_csv(csv_path):
    """
    Parses annotator CSV file into a clean DataFrame.

    Args:
        csv_path (str): Path to annotator CSV file

    Returns:
        pd.DataFrame: DataFrame with columns ['review_id', 'review', 'label']
                     where label is one of: 'Positive', 'Negative', 'Neutral'

    Note:
        - Look for relevant column names in the CSV file
        - If column names differ, the function will try to map them appropriately
        - Finally, return with two columns 'review' and 'label'
    """
    # TODO: Load CSV file using pd.read_csv()
    df = pd.read_csv(csv_path)


    # TODO: Check and map column names if needed
    review_cols = ['review', 'text', 'comment', 'review_text', 'sentence']
    label_cols = ['label', 'sentiment', 'annotation', 'rating', 'class']

    for col in review_cols:
        if col in df.columns:
            df['review'] = df[col]
            break
    for col in label_cols:
        if col in df.columns:
            df['label'] = df[col]
            break

    return df[['review', 'label']]

In [ ]:
# TODO: Parse CSV files (replace with actual file paths)

df_a = parse_annotator_csv('Annotator_A.csv')[:100]
df_b = parse_annotator_csv('Annotator_B.csv')[:100]
df_c = parse_annotator_csv('Annotator_C.csv')[:100]

# Adding id for each review
df_a['review_id'] = range(1, len(df_a) + 1)
df_b['review_id'] = range(1, len(df_b) + 1)
df_c['review_id'] = range(1, len(df_c) + 1)


# Display sample data

print("Annotator A:")
display(df_a.head())
print("\nAnnotator B:")
display(df_b.head())
print("\nAnnotator C:")
display(df_c.head())

Annotator A:


,review,label,review_id
0,This movie is a triumph in every sense. Highly...,Positive,1
1,I have never been so bored in my life. The sco...,Negative,2
2,I was completely blown away by this film. The ...,Positive,3
3,The trailer was better than the movie. The act...,Negative,4
4,Middle of the road entertainment. Visually it'...,Neutral,5



Annotator B:


,review,label,review_id
0,This movie is a triumph in every sense. Highly...,Positive,1
1,I have never been so bored in my life. The sco...,Negative,2
2,I was completely blown away by this film. The ...,Positive,3
3,The trailer was better than the movie. The act...,Negative,4
4,Middle of the road entertainment. Visually it'...,Neutral,5



Annotator C:


,review,label,review_id
0,This movie is a triumph in every sense. Highly...,Positive,1
1,I have never been so bored in my life. The sco...,Negative,2
2,I was completely blown away by this film. The ...,Positive,3
3,The trailer was better than the movie. The act...,Negative,4
4,Middle of the road entertainment. Visually it'...,Neutral,5


### Part 1.2: Implement Fleiss' Kappa from Scratch

Measure inter-annotator agreement using Fleiss' Kappa statistic.
Implement the formula from scratch and compare with statsmodels implementation.

In [ ]:
def fleiss_kappa_scratch(rating_matrix):
    """
    Computes Fleiss' Kappa for multiple raters from scratch.

    Args:
        rating_matrix (np.array): A Count Matrix of shape (N, k).
                                  - N = number of items (rows)
                                  - k = number of categories (columns)
                                  - Element [i, j] = Count of raters who assigned category j to iPtem i.
                                  Example:
                                    [[0, 0, 3],   # Item 0: All 3 raters said Category 2
                                     [1, 2, 0]]   # Item 1: 1 rater said Cat 0, 2 said Cat 1


    Returns:
        float: Kappa score (ranges from -1 to 1, where 1 = perfect agreement)

    Formula:
        κ = (P_bar - P_e_bar) / (1 - P_e_bar)

        where:
        - P_bar = (1/N) * Σ(P_i) = average proportion of agreement across all items
        - P_i = (1/(n*(n-1))) * Σ(k_ij * (k_ij - 1)) for item i
        - P_e_bar = Σ(p_j^2) = expected agreement by chance
        - p_j = proportion of all assignments to category j

    Note:
        - N = number of items (samples)
        - n = number of raters per item (should be constant)
        - k_ij = number of raters who assigned category j to item i
    """
    # TODO: Calculate P_bar (observed agreement), P_e_bar (expected agreement by chance), Apply the formula: κ = (P_bar - P_e_bar) / (1 - P_e_bar)
    N,k = rating_matrix.shape
    n = np.sum(rating_matrix[0]) # Assuming constant number of raters for every item

    P_i = []
    for i in range(N):
      k_ij = rating_matrix[i]
      P_i.append((1/(n*(n-1))) * np.sum(k_ij * (k_ij - 1)))

    P_j = np.sum(rating_matrix, axis=0) / (N * n)
    P_bar = np.mean(P_i)
    P_e_bar = np.sum(P_j**2)

    kappa = (P_bar - P_e_bar) / (1 - P_e_bar)

    return kappa



In [ ]:
def prepare_rating_matrix(df_a, df_b, df_c):
    """
    Converts three DataFrames into a rating matrix for Fleiss' Kappa calculation.

    Args:
        df_a, df_b, df_c: DataFrames with columns ['review_id', 'review', 'label']

    Returns:
        np.array: Rating matrix of shape (N_samples, N_categories)
                  where categories are ['Negative', 'Neutral', 'Positive']
    """
    # TODO: Merge the three DataFrames on review
    # Hint: Use pd.merge() or pd.concat() with proper keys
    df_a = df_a[['review_id', 'label']].rename(columns={'label': 'rater_a'})
    df_b = df_b[['review_id', 'label']].rename(columns={'label': 'rater_b'})
    df_c = df_c[['review_id', 'label']].rename(columns={'label': 'rater_c'})


    merged_df = pd.merge(df_a, df_b, on='review_id', how='inner')
    merged_df = pd.merge(merged_df, df_c, on='review_id', how='inner')

    # TODO: Return numpy array of shape (N_samples, 3)
    # Order: [Negative_count, Neutral_count, Positive_count] for each row

    arr = np.zeros((len(merged_df), 3))
    cols = ['rater_a', 'rater_b', "rater_c"]
    for i, row in merged_df.iterrows():
      for col in cols:
        if row[col] == 'Negative':
          arr[i][0] += 1
        if row[col] == 'Neutral':
          arr[i][1] +=1
        if row[col] == 'Positive':
          arr[i][2] +=1
    return arr

# TODO: Prepare rating matrix and calculate Fleiss' Kappa
rating_matrix = prepare_rating_matrix(df_a, df_b, df_c)
kappa_scratch = fleiss_kappa_scratch(rating_matrix)


# TODO: Use statsmodels to calculate Fleiss' Kappa
kappa_statsmodels = fleiss_kappa(rating_matrix) # This came from statmodels

# TODO: Print the difference between the two implementations
print("Kappa calculated from scratch:", kappa_scratch)
print("Kappa calculated from statsmodels:", kappa_statsmodels)

print("\nDifference:", abs(kappa_scratch - kappa_statsmodels))

Kappa calculated from scratch: 0.8504794967861684
Kappa calculated from statsmodels: 0.8504794967861684

Difference: 0.0


In [ ]:
# =========================
# STEP 1: Identify Conflicts
# =========================

# Merge all three annotators with review text
df_a_ = df_a[['review_id', 'review', 'label']].rename(columns={'label': 'rater_a'})
df_b_ = df_b[['review_id', 'label']].rename(columns={'label': 'rater_b'})
df_c_ = df_c[['review_id', 'label']].rename(columns={'label': 'rater_c'})

merged_full = df_a_.merge(df_b_, on='review_id').merge(df_c_, on='review_id')

# Conflict = NOT unanimous
conflicts = merged_full[
    ~((merged_full['rater_a'] == merged_full['rater_b']) &
        (merged_full['rater_b'] == merged_full['rater_c']) )
]

print(f"\nTotal Conflicting Reviews: {len(conflicts)}")

# =========================
# STEP 2: Print 5 Samples
# =========================

print("\nSample Conflicting Reviews:\n")

for i, row in conflicts.head(5).iterrows():
    print("Review:")
    print(row['review'])
    print("Annotator A:", row['rater_a'])
    print("Annotator B:", row['rater_b'])
    print("Annotator C:", row['rater_c'])
    print("-" * 80)


Total Conflicting Reviews: 14

Sample Conflicting Reviews:

Review:
The acting is fine, but the story arc is a bit formulaic. It was a decent way to kill an afternoon.
Annotator A: Negative
Annotator B: Positive
Annotator C: Neutral
--------------------------------------------------------------------------------
Review:
Visually it's fine, but the editing is just adequate. It was a decent way to kill an afternoon.
Annotator A: Neutral
Annotator B: Positive
Annotator C: Positive
--------------------------------------------------------------------------------
Review:
The CGI was okay, but forgettable. I walked in with low expectations and they were met.
Annotator A: Positive
Annotator B: Neutral
Annotator C: Neutral
--------------------------------------------------------------------------------
Review:
Great concept, but the execution was clichéd. Ideally, this should have been great, but something was off.
Annotator A: Negative
Annotator B: Neutral
Annotator C: Neutral
---------------

### Part 1.3: Conflict Resolution

Identify conflicts where annotators disagree and resolve them using majority vote.
For complete ties (all three differ), default to 'Neutral'.

In [ ]:
from multiprocessing.spawn import prepare
def resolve_conflicts(df_a, df_b, df_c):
    """
    Merges annotations from 3 annotators, resolves disagreements using Majority Vote,
    and handles complete ties by defaulting to 'Neutral'.

    Args:
        df_a, df_b, df_c: DataFrames from each annotator with columns ['review', 'label']

    Returns:
        pd.DataFrame: Final DataFrame with resolved labels (gold standard)
                     Columns: ['review', 'label']

    Logic:
        - Majority Vote: If 2 annotators agree, use their label
        - Tie-Breaker: If all 3 differ (e.g., Positive vs. Negative vs. Neutral), assign 'Neutral'
    """
    rating_matrix = prepare_rating_matrix(df_a, df_b, df_c)

    arr = []


    """
    for i in range(len(rating_matrix)):
      if rating_matrix[i][0] > rating_matrix[i][1] and rating_matrix[i][0] >rating_matrix[i][2]:
        arr.append([df_a.iloc[i]['review'], "Negative"])
      elif rating_matrix[i][1] > rating_matrix[i][0] and rating_matrix[i][1] > rating_matrix[i][2]:
        arr.append([df_a.iloc[i]['review'], "Neutral"])
      elif rating_matrix[i][2] > rating_matrix[i][0] and rating_matrix[i][2] > rating_matrix[i][1]:
        arr.append([df_a.iloc[i]['review'], "Positive"])
      else:
        arr.append([df_a.iloc[i]['review'], "Neutral"])

    return pd.DataFrame(arr, columns=["review", 'label'])
    """
    df_a = df_a[['review_id', 'review', 'label']].rename(columns={'label': 'a'})
    df_b = df_b[['review_id', 'label']].rename(columns={'label': 'b'})
    df_c = df_c[['review_id', 'label']].rename(columns={'label': 'c'})
    merged = df_a.merge(df_b, on='review_id', how='inner').merge(df_c, on='review_id', how='inner')


    for _, row in merged.iterrows():
      labels = [row['a'], row['b'], row['c']]
      count = pd.Series(labels).value_counts()

      if count.iloc[0] >=2:
        final_label = count.index[0]
      else:
        final_label = "Neutral"
      arr.append([row['review'], final_label])

    return pd.DataFrame(arr, columns=["review", 'label'])

In [ ]:
# TODO: Resolve conflicts and create gold standard
gold_standard = resolve_conflicts(df_a, df_b, df_c)


# TODO: Display 5 examples of conflicting reviews (if <5 reviews, show all)
# Show what A, B, and C each said, and the final resolved label
df_a_ = df_a[['review_id', 'review', 'label']].rename(columns={'label': 'A'})
df_b_ = df_b[['review_id', 'label']].rename(columns={'label': 'B'})
df_c_ = df_c[['review_id', 'label']].rename(columns={'label': 'C'})

merged = df_a_.merge(df_b_, on='review_id', how='inner').merge(df_c_, on='review_id', how='inner')

conflicts = merged[~((merged['A'] == merged['B']) & (merged['B'] == merged['C']))]

conflicts = conflicts.merge(gold_standard, on='review', how='left').rename(columns={'label': 'Final'})

print("Examples of conflicting reviews:\n")

display_cols = ['review', 'A', 'B', 'C', "Final"]

# Show up to 5 conflicts
display(conflicts[display_cols].head(5))

print ("Gold standard csv ")
display(gold_standard.head(5))
# TODO: Save gold standard to CSV
gold_standard.to_csv('gold_standard_100.csv', index=False)

Examples of conflicting reviews:



,review,A,B,C,Final
0,"The acting is fine, but the story arc is a bit...",Negative,Positive,Neutral,Neutral
1,"Visually it's fine, but the editing is just ad...",Neutral,Positive,Positive,Positive
2,"The CGI was okay, but forgettable. I walked in...",Positive,Neutral,Neutral,Neutral
3,"Great concept, but the execution was clichéd. ...",Negative,Neutral,Neutral,Neutral
4,"I tried to like it, I really did, but it faile...",Negative,Negative,Neutral,Negative


Gold standard csv 


,review,label
0,This movie is a triumph in every sense. Highly...,Positive
1,I have never been so bored in my life. The sco...,Negative
2,I was completely blown away by this film. The ...,Positive
3,The trailer was better than the movie. The act...,Negative
4,Middle of the road entertainment. Visually it'...,Neutral


## Task 2: Weak Supervision (The "Lazy" Labeler) (6 Marks)

**Objective:** Label the next 200 reviews programmatically to save time.

### Part 2.1: Heuristic Development

Analyze patterns in the gold standard and write at least 3 heuristic functions.
Apply them to the remaining 200 unlabeled reviews.

In [ ]:
# Constants for labeling functions
POSITIVE = 1
NEGATIVE = 0
NEUTRAL = 2
ABSTAIN = -1

# TODO: Load gold standard to analyze patterns
gold_df = pd.read_csv("gold_standard_100.csv")

# TODO: Analyze patterns (e.g., common positive/negative words, review length, etc.)
# ==============================
# Analyze Patterns in Gold Data
# ==============================

# Basic distribution
print("Label Distribution:\n")
print(gold_df['label'].value_counts())
print("\nPercentage Distribution:\n")
print(gold_df['label'].value_counts(normalize=True) * 100)


# ------------------------------
# Review Length Analysis
# ------------------------------

gold_df['word_count'] = gold_df['review'].apply(lambda x: len(x.split()))

print("\nAverage Review Length by Label:\n")
print(gold_df.groupby('label')['word_count'].mean())


# ------------------------------
# Common Words Per Label
# ------------------------------

from collections import Counter

def top_words(label, n=20):
    text = " ".join(gold_df[gold_df['label'] == label]['review'].str.lower())
    words = re.findall(r'\b[a-z]{3,}\b', text)  # ignore very short words
    return Counter(words).most_common(n)

print("\nTop Positive Words:")
print(top_words("Positive"))

print("\nTop Negative Words:")
print(top_words("Negative"))

print("\nTop Neutral Words:")
print(top_words("Neutral"))


# ------------------------------
# Strong Sentiment Indicators
# ------------------------------

# Check frequency of exclamation marks
gold_df['exclamation'] = gold_df['review'].apply(lambda x: "!" in x)

print("\nExclamation Usage by Label:")
print(gold_df.groupby('label')['exclamation'].mean() * 100)


# ------------------------------
# Uppercase Usage (Emphasis)
# ------------------------------

gold_df['uppercase_ratio'] = gold_df['review'].apply(
    lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
)

print("\nAverage Uppercase Ratio by Label:")
print(gold_df.groupby('label')['uppercase_ratio'].mean())

# This will help you design effective heuristics

Label Distribution:

label
Neutral     46
Positive    30
Negative    24
Name: count, dtype: int64

Percentage Distribution:

label
Neutral     46.0
Positive    30.0
Negative    24.0
Name: proportion, dtype: float64

Average Review Length by Label:

label
Negative    14.875000
Neutral     18.130435
Positive    16.600000
Name: word_count, dtype: float64

Top Positive Words:
[('the', 23), ('this', 22), ('was', 12), ('and', 12), ('for', 10), ('movie', 9), ('every', 7), ('wow', 6), ('much', 6), ('triumph', 5), ('sense', 5), ('perfectly', 5), ('balances', 5), ('humor', 5), ('drama', 5), ('from', 5), ('don', 5), ('miss', 5), ('one', 5), ('lead', 4)]

Top Negative Words:
[('the', 22), ('was', 8), ('complete', 8), ('never', 5), ('this', 5), ('misfire', 5), ('through', 5), ('have', 4), ('been', 4), ('bored', 4), ('life', 4), ('frankly', 4), ('all', 4), ('movie', 4), ('zero', 4), ('stars', 4), ('could', 4), ('give', 4), ('like', 4), ('from', 4)]

Top Neutral Words:
[('the', 52), ('was', 27), ('bu

### Part 2.2: Snorkel Labeling Functions

Wrap your heuristics as Snorkel @labeling_function decorators.
Each function should return POSITIVE (1), NEGATIVE (0), NEUTRAL (2), or ABSTAIN (-1).

In [ ]:
@labeling_function()
def lf_keyword_great(x):
    """
    Example labeling function: Check if "great" appears in the review.
    Returns POSITIVE if found, otherwise ABSTAIN.
    """
    # TODO: Check if "great" (case-insensitive) is in x.review
    # Return POSITIVE if found, ABSTAIN otherwise
    if "great" in x.review.lower():
        return POSITIVE
    return ABSTAIN


@labeling_function()
def lf_short_review(x):
    """
    Label based on review length.
    Very short reviews might be neutral or indicate lack of engagement.
    """
    # TODO: Implement logic based on review length
    # Return appropriate label (NEUTRAL for very short, or ABSTAIN)
    if len(x.review.split()) < 4:
        return NEUTRAL
    return ABSTAIN

@labeling_function()
def lf_regex_bad(x):
    """
    Use regex to find negative patterns.
    Look for words like "horrible", "terrible", "awful", etc.
    """
    # TODO: Use regex or string matching to find negative keywords
    # Return NEGATIVE if found, ABSTAIN otherwise
    if re.search(r"\b(horrible|terrible|awful|worst|boring|bad)\b", x.review.lower()):
        return NEGATIVE
    return ABSTAIN

# TODO: Write at least 3 more labeling functions (minimum 6 total)

@labeling_function()
def lf_positive_words(x):
    if re.search(r"\b(amazing|excellent|fantastic|wonderful|love|best)\b", x.review.lower()):
        return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_neutral_words(x):
    if re.search(r"\b(okay|fine|average|normal|mediocre)\b", x.review.lower()):
        return NEUTRAL
    return ABSTAIN

@labeling_function()
def lf_exclamation_sentiment(x):
    if "!" in x.review:
        if re.search(r"\b(love|great|amazing|fantastic)\b", x.review.lower()):
            return POSITIVE
        if re.search(r"\b(hate|terrible|awful|worst)\b", x.review.lower()):
            return NEGATIVE
    return ABSTAIN



### Part 2.3: Apply Labeling Functions and Analyze Coverage

Apply all labeling functions to the 200 unlabeled reviews and calculate coverage and conflict rates.

In [ ]:
def analyze_weak_labels(L_matrix, lfs):
    """
    Prints Coverage and Conflict statistics for the Labeling Functions.

    Args:
        L_matrix (np.array): Label matrix of shape (N_samples, N_functions)
                            Each column represents one labeling function's outputs
                            Values: POSITIVE (1), NEGATIVE (0), NEUTRAL (2), ABSTAIN (-1)
        lfs: List of labeling functions (for display names)

    Metrics to calculate:
        - Coverage: Percentage of non-abstain votes per LF
        - Conflict Rate: Percentage of samples where LFs disagree
    """
    # TODO: Calculate coverage for each labeling function
    # Coverage = (number of non-abstain votes) / (total samples) * 100


    # TODO: Calculate conflict rate
    # Conflict occurs when multiple LFs label the same sample differently
    # Conflict Rate = (number of conflicting samples) / (total samples) * 100


    # TODO: Print statistics in a readable format
    # Hint: Use LFAnalysis from snorkel for detailed stats (optional)
    # Or print manually: LF name, Coverage %, Conflicts count

    N, M = L_matrix.shape

    print("Coverage per Labeling Function:\n")

    for i, lf in enumerate(lfs):
        coverage = np.sum(L_matrix[:, i] != ABSTAIN) / N * 100
        print(f"{lf.name}: {coverage:.2f}%")

    # Conflict calculation
    conflict_count = 0

    for row in L_matrix:
        labels = row[row != ABSTAIN]
        if len(set(labels)) > 1:
            conflict_count += 1

    conflict_rate = conflict_count / N * 100

    print(f"\nOverall Conflict Rate: {conflict_rate:.2f}%")



In [ ]:
# TODO: Load the 200 unlabeled reviews (you can load the entire dataset and then filter as per the requirement)
full_data = pd.read_csv("movie_reviews_300.csv")

unlabeled_200 = full_data.iloc[100:300].copy()

print("Unlabeled samples:", len(unlabeled_200))
display(unlabeled_200.head(5))



Unlabeled samples: 200


,review
100,It’s a weird mix of brilliance and stupidity. ...
101,I struggled to sit through the first half. Not...
102,I struggled to sit through the first half. The...
103,I'm honestly still trying to process what I ju...
104,A refreshing take on a tired genre. It perfect...


In [ ]:

# TODO: Apply all labeling functions to create L_matrix
# lfs = [lf_keyword_great, lf_short_review, lf_regex_bad, ...]  # Add all your LFs
# applier = <put your code here>
# L_matrix = <put your code here>

# TODO: Analyze coverage and conflicts
# TODO: Use LFAnalysis for detailed statistics
lfs = [
    lf_keyword_great,
    lf_short_review,
    lf_regex_bad,
    lf_positive_words,
    lf_neutral_words,
    lf_exclamation_sentiment
]

applier = PandasLFApplier(lfs=lfs)
L_matrix = applier.apply(df=unlabeled_200)

analyze_weak_labels(L_matrix, lfs)

# ===============================
# Snorkel LF Detailed Statistics
# ===============================

analysis = LFAnalysis(L=L_matrix, lfs=lfs)

print("\nLabeling Function Summary:")
display(analysis.lf_summary())

print("\nLabeling Function Coverage:")
display(analysis.lf_coverages())

print("\nLabeling Function Overlaps:")
display(analysis.lf_overlaps())

print("\nLabeling Function Conflicts:")
display(analysis.lf_conflicts())


100%|██████████| 200/200 [00:00<00:00, 14155.84it/s]

Coverage per Labeling Function:

lf_keyword_great: 5.00%
lf_short_review: 0.00%
lf_regex_bad: 10.00%
lf_positive_words: 1.50%
lf_neutral_words: 11.50%
lf_exclamation_sentiment: 0.00%

Overall Conflict Rate: 1.00%

Labeling Function Summary:


,j,Polarity,Coverage,Overlaps,Conflicts
lf_keyword_great,0,[1],0.050,0.00,0.00
lf_short_review,1,[],0.000,0.00,0.00
lf_regex_bad,2,[0],0.100,0.00,0.00
lf_positive_words,3,[1],0.015,0.01,0.01
lf_neutral_words,4,[2],0.115,0.01,0.01
lf_exclamation_sentiment,5,[],0.000,0.00,0.00



Labeling Function Coverage:


array([0.05 , 0.   , 0.1  , 0.015, 0.115, 0.   ])


Labeling Function Overlaps:


array([0.  , 0.  , 0.  , 0.01, 0.01, 0.  ])


Labeling Function Conflicts:


array([0.  , 0.  , 0.  , 0.01, 0.01, 0.  ])

### Part 2.4: Majority Vote Adjudication

Use majority vote to generate probabilistic labels (weak labels) for the 200 reviews.
Save the result to `weak_labels_200.csv`.

In [ ]:
# TODO: Train LabelModel to get probabilistic labels

label_model = LabelModel(cardinality=3, verbose=True)
label_model.fit(L_train=L_matrix, n_epochs=500, log_freq=100)

probs = label_model.predict_proba(L_matrix)
weak_labels_numeric = probs.argmax(axis=1)


# TODO: Convert numeric labels to match your label scheme
# Label mapping: 0 -> 'Negative' (or 0), 1 -> 'Positive' (or 1), 2 -> 'Neutral' (or 2), -1 -> 'Abstain'

label_map = {
    0: "Negative",
    1: "Positive",
    2: "Neutral"
}

weak_labels_text = [label_map[label] for label in weak_labels_numeric]

# TODO: Create DataFrame with reviews and weak labels
weak_labels_df = pd.DataFrame({
    "review": unlabeled_200['review'],
    "label": weak_labels_text
})


# TODO: Save to CSV
weak_labels_df.to_csv("weak_labels_200.csv", index=False)
display(weak_labels_df.head(5))
print("weak_labels_200.csv saved successfully.")


100%|██████████| 500/500 [00:00<00:00, 607.09epoch/s]


,review,label
100,It’s a weird mix of brilliance and stupidity. ...,Positive
101,I struggled to sit through the first half. Not...,Negative
102,I struggled to sit through the first half. The...,Negative
103,I'm honestly still trying to process what I ju...,Negative
104,A refreshing take on a tired genre. It perfect...,Negative


weak_labels_200.csv saved successfully.


## Task 3: Active Learning (The Budget Optimizer) (5 Marks)

**Objective:** Simulate cost savings by training a model iteratively.

### Part 3.1: Query Strategy Implementation

Implement Least Confidence and Entropy Sampling from scratch.
These strategies select the most informative samples for labeling.

In [ ]:
def least_confidence_sampling(model, X_pool, n_instances=10):
    """
    Selects samples where the model is least confident (uncertainty sampling).

    Args:
        model: Trained classifier with predict_proba() method
        X_pool: Feature matrix of unlabeled samples
        n_instances: Number of samples to select

    Returns:
        np.array: Indices of selected samples

    Strategy:
        Uncertainty = 1 - max(probability) across all classes
        For 3-class classification: Get probabilities for [Negative, Positive, Neutral]
        Select samples with highest uncertainty (lowest max probability)
    """
    # TODO: Get probability predictions from model
    probas = model.predict_proba(X_pool)

    # TODO: Calculate uncertainty: 1 - max(probability) for each sample
    uncertainty = 1 - np.max(probas, axis=1)

    # TODO: Select top n_instances samples with highest uncertainty
    query_indices = np.argsort(uncertainty)[-n_instances:][::-1]

    return query_indices

    pass

def entropy_sampling(model, X_pool, n_instances=10):
    """
    Selects samples with highest entropy (information gain).

    Args:
        model: Trained classifier with predict_proba() method
        X_pool: Feature matrix of unlabeled samples
        n_instances: Number of samples to select

    Returns:
        np.array: Indices of selected samples

    Strategy:
        Entropy = -sum(p * log(p)) for all classes
        For 3-class classification: Calculate entropy across [Negative, Positive, Neutral] probabilities
        Select samples with highest entropy (most uncertain across all classes)
    """
    # TODO: Get probability predictions from model
    probas = model.predict_proba(X_pool)
    # TODO: Calculate entropy: -sum(p * log(p)) for each sample
    # Add small epsilon (1e-9) to avoid log(0) errors
    epsilon = 1e-9
    entropies = -np.sum(probas * np.log(probas + epsilon), axis=1)
    # TODO: Select top n_instances samples with highest entropy
    query_indices = np.argsort(entropies)[-n_instances:][::-1]

    return query_indices

    pass

def random_sampling(model, X_pool, n_instances=10):
    """
    Baseline strategy: Selects random samples.

    Args:
        model: Not used, but kept for interface consistency
        X_pool: Feature matrix of unlabeled samples
        n_instances: Number of samples to select

    Returns:
        np.array: Randomly selected indices
    """
    # TODO: Randomly select n_instances indices from X_pool
    n_samples = X_pool.shape[0]
    query_indices = np.random.choice(n_samples, size=n_instances, replace=False)
    return query_indices

    pass

### Part 3.2: Data Processing and Setup

Load the gold standard (seed) and weak labels (pool).
Create a static test set from the pool for evaluation.
Vectorize text data using TF-IDF.

In [ ]:
def load_and_process_data():
    """
    Loads and processes data for active learning.

    Returns:
        Tuple: (X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer)
               All X are feature matrices, all y are label arrays
               vectorizer is returned for later use on LLM data

    Note:
        - Seed: gold_standard_100.csv (100 labeled reviews)
        - Pool: weak_labels_200.csv (200 reviews, labels treated as hidden for simulation)
        - Test: Hold out 50 samples from pool (weak labels) for static evaluation
        - We use 3-class classification: Positive (1), Negative (0), Neutral (2)
        - Uncertainty metrics use probability scores across all three classes:
          * Least Confidence: 1 - max(probabilities) across all classes
          * Entropy: -sum(p * log(p)) for all three classes
    """

    df_seed = pd.read_csv('gold_standard_100.csv')
    df_pool_full = pd.read_csv('weak_labels_200.csv')

    # Ensure both have 'review' column
    if 'review' not in df_seed.columns:
        raise ValueError("gold_standard_100.csv must have 'review' column")
    if 'review' not in df_pool_full.columns:
        raise ValueError("weak_labels_200.csv must have 'review' column")

    # Handle both 'label' and 'sentiment' column names
    label_col_seed = 'label' if 'label' in df_seed.columns else 'sentiment'
    label_col_pool = 'label' if 'label' in df_pool_full.columns else 'sentiment'

    # Map text labels to numeric: Positive=1, Negative=0, Neutral=2
    label_mapping = {
        'Positive': 1, 'positive': 1, 'POSITIVE': 1,
        'Negative': 0, 'negative': 0, 'NEGATIVE': 0,
        'Neutral': 2, 'neutral': 2, 'NEUTRAL': 2
    }

    # Convert seed labels
    if df_seed[label_col_seed].dtype == 'object':
        df_seed['sentiment_numeric'] = df_seed[label_col_seed].map(label_mapping)
        if df_seed['sentiment_numeric'].isna().any():
            raise ValueError(f"Unknown labels in seed data: {df_seed[df_seed['sentiment_numeric'].isna()][label_col_seed].unique()}")
    else:
        df_seed['sentiment_numeric'] = df_seed[label_col_seed].values

    # Convert pool labels
    if df_pool_full[label_col_pool].dtype == 'object':
        df_pool_full['sentiment_numeric'] = df_pool_full[label_col_pool].map(label_mapping)
        if df_pool_full['sentiment_numeric'].isna().any():
            raise ValueError(f"Unknown labels in pool data: {df_pool_full[df_pool_full['sentiment_numeric'].isna()][label_col_pool].unique()}")
    else:
        df_pool_full['sentiment_numeric'] = df_pool_full[label_col_pool].values

    # Create static test set (hold out 50 samples from pool)
    df_pool, df_test = train_test_split(df_pool_full, test_size=50, random_state=42)

    # Vectorize text data using TfidfVectorizer
    # Fit vectorizer on ALL text (seed + pool + test) to ensure consistent dimensions
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    all_text = pd.concat([df_seed['review'], df_pool['review'], df_test['review']])
    vectorizer.fit(all_text)

    # Transform datasets to feature matrices
    X_seed = vectorizer.transform(df_seed['review']).toarray()
    X_pool = vectorizer.transform(df_pool['review']).toarray()
    X_test = vectorizer.transform(df_test['review']).toarray()

    # Extract numeric labels
    y_seed = df_seed['sentiment_numeric'].values
    y_pool = df_pool['sentiment_numeric'].values
    y_test = df_test['sentiment_numeric'].values

    # Return all datasets and vectorizer
    return X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer

# TODO: uncomment below codes, to use these variables further
X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer = load_and_process_data()

print(f"Seed Size: {len(y_seed)}")
print(f"Pool Size: {len(y_pool)} (Available for querying)")
print(f"Test Size: {len(y_test)} (Held out for evaluation)")

### Part 3.3: Active Learning Loop

Implement the iterative active learning loop:
1. Train model on current training set
2. Query uncertain samples from pool
3. "Label" them (reveal ground truth)
4. Add to training set and retrain
5. Log test accuracy

In [ ]:
def run_active_learning_loop(X_seed, y_seed, X_pool, y_pool, X_test, y_test,
                             strategy_func, steps=5, batch_size=10):
    """
    Simulates the active learning loop (matches lab approach).

    Args:
        X_seed, y_seed: Initial training data (seed set)
        X_pool, y_pool: Unlabeled pool (y_pool is hidden, revealed during query)
        X_test, y_test: Static test set for evaluation
        strategy_func: Function that selects samples (e.g., least_confidence_sampling)
                      Signature: strategy_func(model, X_pool, n_instances) -> indices
        steps: Number of iterations
        batch_size: Number of samples to query per iteration

    Returns:
        Tuple: (n_labels_history, accuracy_history)
               Lists tracking number of labels and test accuracy over iterations
    """
    # TODO: Initialize training set with seed data
    X_train = X_seed.copy()
    y_train = y_seed.copy()

    # TODO: Create working copies of pool (we'll remove samples as we query them)
    X_pool_curr = X_pool.copy()
    y_pool_curr = y_pool.copy()

    # TODO: Initialize empty lists to track progress (accuracy_history, n_labels_history)
    accuracy_history = []
    n_labels_history = []

    # Train initial model on seed data
    model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
    model.fit(X_train, y_train)

    # TODO: Evaluate initial model and log results
    y_pred = model.predict(X_test)
    initial_acc = accuracy_score(y_test, y_pred)
    accuracy_history.append(initial_acc)
    n_labels_history.append(len(y_train))

    # TODO: Iterative loop (repeat 'steps' times):
    #   for i in range(steps):
    #       1. Query: Use strategy_func(model, X_pool_curr, batch_size) to get indices
    #       2. "Label": Reveal ground truth: y_new = y_pool_curr[query_indices]
    #       3. Add to training set: use np.vstack() to add new samples
    #       4. Remove from pool: use np.delete() to remove queried samples
    #       5. Retrain model: use model.fit() to update the model
    #       6. Evaluate on test set, get accuracy
    #       7. Log: accuracy_history.append(acc), n_labels_history.append(len(y_train))

    for i in range(steps):
        # 1. Query
        query_indices = strategy_func(model, X_pool_curr, batch_size)

        # 2. "Label"
        X_new = X_pool_curr[query_indices]
        y_new = y_pool_curr[query_indices]

        # 3. Add to training set
        X_train = np.vstack((X_train, X_new))
        y_train = np.hstack((y_train, y_new))

        # 4. Remove from pool
        X_pool_curr = np.delete(X_pool_curr, query_indices, axis=0)
        y_pool_curr = np.delete(y_pool_curr, query_indices, axis=0)

        # 5. Retrain model
        model.fit(X_train, y_train)

        # 6. Evaluate
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        # 7. Log
        accuracy_history.append(acc)
        n_labels_history.append(len(y_train))

    # TODO: Return history lists
    return n_labels_history, accuracy_history


# TODO: Run active learning with least confidence strategy
n_labels_lc, accuracy_lc = run_active_learning_loop(
    X_seed, y_seed, X_pool, y_pool, X_test, y_test,
    strategy_func=least_confidence_sampling,
    steps=5,
    batch_size=10
)

### Part 3.4: Visualization and Comparison

Plot learning curves comparing Active Learning vs. Random Sampling.

In [ ]:
# TODO: Run active learning with random sampling (baseline)

# Run active learning with entropy sampling strategy
print("\n" + "="*70)
print("Running active learning with entropy sampling strategy")
print("="*70)

n_labels_entropy, accuracy_entropy = run_active_learning_loop(
    X_seed, y_seed, X_pool, y_pool, X_test, y_test,
    strategy_func=entropy_sampling,
    steps=5, batch_size=10
)

# RUN baseline: random sampling
print("\n" + "="*70)
print("Running Baseline: Random Sampling")
print("="*70)

n_labels_random, accuracy_random = run_active_learning_loop(
    X_seed, y_seed, X_pool, y_pool, X_test, y_test,
    strategy_func=random_sampling,
    steps=5, batch_size=10
)


# TODO: Plot learning curves of active learning and random sampling wrt to number of samples
plt.figure(figsize=(12,6))
plt.plot(n_labels_lc, accuracy_lc, marker='o', linewidth=2,label='Active Learning (Least Confidence)', color='blue')
plt.plot(n_labels_entropy, accuracy_entropy, marker='s', linewidth=2,label='Active Learning (Entropy)', color='green')
plt.plot(n_labels_random, accuracy_random, marker='^', linewidth=2,label='Baseline: Random Sampling', color='red', linestyle='--')
plt.xlabel('Number of Samples (Labels)', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Active Learning vs Random Sampling: Learning Curves',fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO: Print comparison summary for active learning and random sampling final accuracy
print("\n" + "="*70)
print("FINAL COMPARISON SUMMARY")
print("="*70)

print(f"\nFinal Test Accuracy After {len(n_labels_lc)-1} Iterations:")
print(f"  Least Confidence:  {accuracy_lc[-1]:.4f} ({n_labels_lc[-1]} labels)")
print(f"  Entropy Sampling:  {accuracy_entropy[-1]:.4f} ({n_labels_entropy[-1]} labels)")
print(f"  Random Sampling:   {accuracy_random[-1]:.4f} ({n_labels_random[-1]} labels)")



## Task 4: AI vs. AI (LLM & Noise Detection) (3 Marks)

**Objective:** Use LLMs for bulk labeling and detect hallucinations.

**Note:**

- Make an account at [open-router](https://openrouter.ai/) and get the API key.
- Use `google/gemini-2.5-flash-lite` (free tier) model as your LLM. Read the documentation on how to use it [here](https://openrouter.ai/google/gemini-2.5-flash-lite/api)
- Set environment variable using .env file and paste your API key in it.

### Part 4.1: LLM Pipeline with Few-Shot Prompting

Design a few-shot prompt with 3 examples from gold standard.
Send remaining unlabeled samples (~150) to Gemini API for labeling.

In [ ]:
df = pd.read_csv("movie_reviews_300.csv")

df

In [ ]:

import os
import time
import json
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('OPENROUTER_API_KEY')
SITE_URL = "http://localhost:8000"  #for OpenRouter rankings
SITE_NAME = "Student Lab Assignment"

MODEL_NAME = "google/gemini-2.5-flash-lite"



if not API_KEY:
    print("Warning: OPENROUTER_API_KEY not found. Please check your .env file.")


def generate_few_shot_prompt(review_text, examples):
    """
    Constructs a few-shot prompt with 3 gold examples + target review.
    """

    prompt = "You are a sentiment classifier.\n"
    prompt += "Possible labels: Positive, Negative, Neutral.\n"
    prompt += "Return only one word.\n\n"

    for ex in examples:
        prompt += f"Review: {ex['review']}\n"
        prompt += f"Label: {ex['label']}\n\n"

    prompt += f"Review: {review_text}\n"
    prompt += "Label:"

    return prompt


def query_openrouter(review_text, examples):
    """
    Sends request to OpenRouter API with retry logic and parsing.
    """
    url = "https://openrouter.ai/api/v1/chat/completions"

    # TODO: Set up headers:
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": SITE_URL,
        "X-Title": SITE_NAME
    }

    # TODO: Generate prompt using generate_few_shot_prompt()
    prompt = generate_few_shot_prompt(review_text, examples)

    # TODO: Create payload dictionary:
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    # TODO: Implement retry logic:
    for _ in range(3):
        response = requests.post(url, headers=headers, data=json.dumps(payload))

        if response.status_code == 200:

            # TODO: Parse successful response:
            result = response.json()
            text = result["choices"][0]["message"]["content"].strip()
            label = text.split()[0]
            return label

        elif response.status_code == 429:
            print("Rate limit hit")
            time.sleep(5)
        else:
            print("Error:", response.status_code)
            return None

    return None


# --- MAIN EXECUTION ---

# TODO: Load gold standard examples for few-shot prompting
df = pd.read_csv(("gold_standard_100.csv"))
examples = df.head(3)[["review", "label"]].to_dict("records")

# TODO: Load remaining unlabeled reviews (~150, select last 150 from movie_reviews_300.csv)
unlabeled = df.tail(150)

# TODO: Query OpenRouter for each review
# Handle free tier requests per minute (RPM) limit of ~15
results = []


for review in unlabeled["review"]:
    label = query_openrouter(review, examples)

    results.append({
        "review": review,
        "label": label
    })

    time.sleep(4)

# TODO: Save LLM labels, in csv format with 'review' and 'label' columns
output_df = pd.DataFrame(results)
output_df.to_csv("llm_labels_150.csv", index=False)




### Part 4.2: Noise Hunting (Cleanlab Logic)

Train a Logistic Regression model on LLM-labeled data.
Identify "High Confidence Disagreements" where the model is very confident (>0.80) but disagrees with the LLM label.

In [ ]:
import numpy as np
import pandas as pd

def find_label_errors(llm_labels, model_probs, review_texts, threshold=0.90):
    """
    Detects high-confidence disagreements between model predictions and LLM labels.
    """

    # TODO: Get model predictions from probabilities
    preds = np.argmax(model_probs, axis=1)

    # TODO: Get model confidence (max probability) for each sample
    confidences = np.max(model_probs, axis=1)

    # TODO: Convert llm_labels to numeric if they are strings
    mapping = {"Negative": 0, "Positive": 1, "Neutral": 2}
    llm_numeric = [
        mapping[label] if isinstance(label, str) else label
        for label in llm_labels
    ]

    # TODO: Find disagreements
    disagreement_mask = (preds != llm_numeric) & (confidences > threshold)

    # TODO: Create list of suspicious reviews
    suspicious = []
    for i, flag in enumerate(disagreement_mask):
        if flag:
            suspicious.append({
                "index": i,
                "text": review_texts[i],
                "llm_label": llm_numeric[i],
                "model_pred": preds[i],
                "confidence": confidences[i]
            })

    # TODO: Sort by confidence (highest first)
    suspicious = sorted(suspicious, key=lambda x: x["confidence"], reverse=True)

    # TODO: Return
    return suspicious


# ---------------------------------------------------
# MAIN
# ---------------------------------------------------

# TODO: Load LLM labels in dataframe
df = pd.read_csv("llm_labeled_reviews.csv")

review_texts = df["review"].tolist()
llm_labels = df["label"].tolist()


# TODO: Vectorize LLM-labeled reviews (use same vectorizer from Task 3)
X = vectorizer.transform(review_texts)


# TODO: Train Logistic Regression on LLM-labeled data
model.fit(X, llm_labels)


# TODO: Get probabilities on the same data
model_probs = model.predict_proba(X)


# TODO: Find label errors
errors = find_label_errors(llm_labels, model_probs, review_texts)


# TODO: Print top 5 suspicious reviews
print("\n Suspicious Reviews:\n")

for item in errors[:5]:
    print("Index:", item["index"])
    print("Text:", item["text"])
    print("LLM Label:", item["llm_label"])
    print("Model Prediction:", item["model_pred"])
    print("Confidence:", round(item["confidence"], 3))
    print("-" * 40)

## Deliverables

**Submission Checklist:**
- [ ] Completed Jupyter Notebook with all tasks (Tasks 1-4)
- [ ] Include your label-studio annotation interface screenshot.
- [ ] gold_standard_100.csv
- [ ] weak_labels_200.csv
- [ ] llm_labels_150.json